In [ ]:
from glob import glob
from pathlib import Path
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors
import numpy as np
from shapely.ops import transform

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"
targets_path = "power/targets.geoparquet"  # within results_dir
country_polygons_path = "input/admin-boundaries/admin-level-0.geoparquet"  # within results_dir

# Output paths
plot_dir = Path("../plots")

# Specify group members (GCM variants) of scenarios
scenario_atmosphere_groups = {
    "IRIS baseline": ["IRIS_PRESENT"],
    "IRIS SSP5-8.5 2050": ["IRIS_SSP5-2050"],
    "STORM baseline": ["STORM_constant"],
    "STORM RCP8.5 2050": [
        "STORM_CNRM-CM6-1-HR",
        "STORM_CMCC-CM2-VHR4",
        "STORM_EC-Earth3P-HR",
        "STORM_HadGEM3-GC31-HM"
    ],
    "CHAZ baseline": [
        "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010",
        "CHAZ_SSP-585_GCM-CESM2_epoch-2010",
        "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010",
        "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010",
        "CHAZ_SSP-585_GCM-MIROC6_epoch-2010",
    ],
    "CHAZ SSP5-8.5 2050": [
        "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050",
        "CHAZ_SSP-585_GCM-CESM2_epoch-2050",
        "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050",
        "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050",
        "CHAZ_SSP-585_GCM-MIROC6_epoch-2050",
    ],   
    "MIT baseline": [
        "emanuel_ssp-585_gcm-cesm2_epoch-2005",
        "emanuel_ssp-585_gcm-cnrm6_epoch-2005",
        "emanuel_ssp-585_gcm-ecearth6_epoch-2005",
        "emanuel_ssp-585_gcm-miroc6_epoch-2005",
        "emanuel_ssp-585_gcm-ukmo6_epoch-2005",
    ],
    "MIT SSP5-8.5 2050": [
        "emanuel_ssp-585_gcm-cesm2_epoch-2050",
        "emanuel_ssp-585_gcm-cnrm6_epoch-2050",
        "emanuel_ssp-585_gcm-ecearth6_epoch-2050",
        "emanuel_ssp-585_gcm-miroc6_epoch-2050",
        "emanuel_ssp-585_gcm-ukmo6_epoch-2050",
    ],
}
atmospheres = [item for group in scenario_atmosphere_groups.values() for item in group]
atmosphere_to_scenario = {}
for scenario, atmosphere_set in scenario_atmosphere_groups.items():
    for atmosphere in atmosphere_set:
        atmosphere_to_scenario[atmosphere] = scenario
tc_models = tuple(set(scenario.split()[0] for scenario in scenario_atmosphere_groups.keys()))

In [ ]:
# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)
# Global default threshold, ms-1
default_threshold = 30.0

disruption = []
for atmosphere in atmospheres:
    print(atmosphere, end="\r")
    path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/EAPA_admin-level-2-0.gpq")
    df = gpd.read_parquet(path)
    # Set countries with no calibrated value to the global default
    df["EAPA"] = df[f"{default_threshold:.1f}"]
    valid_countries = thresholds.index.unique()
    df = df[df.ISO_A3.isin(valid_countries)]
    df = df.merge(thresholds[['threshold_ms-1']], left_on='ISO_A3', right_index=True, how='left')
    df['EAPA'] = df.apply(
        lambda row: row[str(row['threshold_ms-1'])] if pd.notna(row['threshold_ms-1']) else np.nan,
        axis=1
    )
    # Drop the now redundant master set of thresholds
    cols_to_keep = ['GID_0', 'GID_1', 'GID_2', 'ISO_A3', 'NAME_0', 'NAME_1', 'NAME_2', 'geometry', 'EAPA']
    # Ideally, I wouldn't fillna here, but if I remove this (and handle the zero division error later)
    # the results are not what I'd expect... so I'm leaving it and not investigating any further for now
    df["EAPA"] = df["EAPA"].fillna(0)
    df = df.loc[:, cols_to_keep].reset_index(drop=True)
    df = df.sort_values("EAPA", ascending=False)
    df["atmosphere"] = atmosphere
    df["GID"] = df.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
    disruption.append(df)

place_cols = ["GID", "GID_0", "GID_1", "GID_2", "ISO_A3", "NAME_0", "NAME_1", "NAME_2", "geometry"]
places = pd.concat(disruption)
places["GID"] = places.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
places = places[place_cols].drop_duplicates().reset_index(drop=True).sort_values("GID")

In [ ]:
all_atmospheres = pd.concat(disruption)
all_atmospheres["scenario"] = all_atmospheres.atmosphere.map(atmosphere_to_scenario)
all_atmospheres["GID"] = all_atmospheres.apply(lambda row: f"{row.GID_0}_{row.GID_1}_{row.GID_2}", axis=1)
all_atmospheres["epoch"] = all_atmospheres.scenario.apply(lambda s: "Baseline" if s.endswith("baseline") else "Future")
data = all_atmospheres.set_index(["GID", "scenario", "atmosphere", "epoch"])[["EAPA"]]

aggregated_groups = []
all_gcm_member_names: set[str] = set()

for scenario, member_names in scenario_atmosphere_groups.items():
    all_gcm_member_names.update(member_names)

    subset = data[data.index.get_level_values("atmosphere").isin(member_names)]
    groupby_region = subset.loc[:, "EAPA"].groupby("GID")

    mean = groupby_region.mean()

    group_df = pd.DataFrame(
        index=pd.MultiIndex.from_product(
            [mean.index, [scenario]], names=["GID", "Scenario"]
        )
    )
    group_df = group_df.join(mean)
    aggregated_groups.append(group_df)

data = pd.concat(aggregated_groups)
data["Epoch"] = data.index.get_level_values("Scenario").map(lambda s: "Baseline" if s.endswith("baseline") else "Future")
data = data.reset_index().set_index(["GID", "Scenario", "Epoch"])

df = gpd.GeoDataFrame(data.join(places.set_index("GID")[["ISO_A3", "geometry"]], how="left"))
# Estimate population connected to grid in each country
targets = gpd.read_parquet(os.path.join(results_dir, targets_path), columns=["population", "iso_a3", "geometry"])
targets["geometry"] = targets.geometry.representative_point()
# Duplicate regions, each with different target
region_to_target = df.reset_index().set_index("GID").sjoin(targets, how="left")
# Group by region, sum target populations to find connected population per region
population = region_to_target[["population"]].groupby(region_to_target.index).sum()
df = df.join(population)
# Calculate a measure normalised by the regional (connected) population
df["EAPA_pc"] = df.EAPA / df.population
df = df[["EAPA", "EAPA_pc", "population", "ISO_A3", "geometry"]]

In [ ]:
def shift_lon(geom):
    def shift(x, y, z=None):
        x = np.where(x < 0, x + 360, x)
        return x, y
    return transform(shift, geom)

# Shift longitude to [0, 360] on unique geometries
pos_long_places = places.reset_index(drop=True)
pos_long_places["geometry"] = pos_long_places["geometry"].apply(shift_lon)
df = gpd.GeoDataFrame(df.drop(columns=["geometry"]).join(pos_long_places.drop(columns=["ISO_A3"]).set_index("GID")))

In [ ]:
STRIDE = 1
MIN_EAPA_PC = 1E-4
to_plot = df.copy()
to_plot = to_plot[~to_plot["EAPA_pc"].isna()]
to_plot = to_plot[to_plot["EAPA_pc"] > MIN_EAPA_PC]
to_plot = to_plot[["EAPA", "EAPA_pc", "geometry"]]
to_plot = to_plot.iloc[::STRIDE]

fig, axes = plt.subplots(4, 1, figsize=(10, 18), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
cmap = "PuBuGn"
norm = matplotlib.colors.LogNorm(
    vmin=to_plot.EAPA_pc.quantile(0.05),
    vmax=to_plot.EAPA_pc.quantile(0.95),
)
baseline_scenarios = sorted([s for s in to_plot.loc[:, :, "Baseline"].index.get_level_values(1).unique() if s.endswith("baseline")])
for i, (ax, scenario) in enumerate(zip(axes, baseline_scenarios)):
    ax.add_feature(cfeature.OCEAN, facecolor="0.97")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.top_labels = False
    gl.bottom_labels = False
    gl.right_labels = False
    to_plot.loc[:, scenario, :].plot(
        "EAPA_pc",
        transform=ccrs.PlateCarree(),
        ax=ax,
        cmap=cmap,
        norm=norm,
        legend=False,
    )
    ax.set_extent([20, 320, -52, 58], crs=ccrs.PlateCarree())
    ax.text(
        0.03, 0.06, scenario.split()[0],
        horizontalalignment='left', verticalalignment='bottom',
        transform=ax.transAxes, fontsize=14
    )
    if i == 0:
        gl.top_labels = True
    if i == len(axes) - 1:
        gl.bottom_labels = True

# Force layout, then manually stack axes to nearly touch
fig.canvas.draw()  # finalize cartopy's aspect-adjusted axes positions

gap = 0.005  # Fraction of figure height between panels
pos0 = axes[0].get_position()
w, h = pos0.width, pos0.height
x0 = pos0.x0
top = pos0.y1  # Keep the top panel where it currently sits

for i, ax in enumerate(axes):
    y1 = top - i * (h + gap)
    y0 = y1 - h
    ax.set_position([x0, y0, w, h])

cax = fig.add_axes([0.95, 0.45, 0.03, 0.2])
sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('EAPA per capita', labelpad=15)

fig.savefig(plot_dir / "eapa_pc_map_by_model.pdf")

In [ ]:
# Baseline -> future scenario pairs, by family
FAMILY_SCENARIOS = {
    "IRIS":  ("IRIS baseline",  "IRIS SSP5-8.5 2050"),
    "STORM": ("STORM baseline", "STORM RCP8.5 2050"),
    "CHAZ":  ("CHAZ baseline",  "CHAZ SSP5-8.5 2050"),
    "MIT":   ("MIT baseline",   "MIT SSP5-8.5 2050"),
}

# GID x Scenario table of EAPA_pc (one value per GID/scenario, ignoring Epoch)
eapa_pc = (
    df["EAPA_pc"]
    .reset_index()
    .pivot_table(index="GID", columns="Scenario", values="EAPA_pc", aggfunc="first")
)

# Sign of change (baseline -> future) for each family: -1, 0, +1, or NaN if either side missing
signs = pd.DataFrame(index=eapa_pc.index)
for family, (baseline, future) in FAMILY_SCENARIOS.items():
    signs[family] = np.sign(eapa_pc[future] - eapa_pc[baseline])

agreement = pd.DataFrame({
    "agreement_score": signs.sum(axis=1, skipna=True),
    "n_models": signs.notna().sum(axis=1),
})

# Bring geometry back in (one geometry per GID)
geom = df.reset_index().drop_duplicates("GID").set_index("GID")["geometry"]
agreement_gdf = agreement.join(geom)
agreement_gdf = gpd.GeoDataFrame(agreement_gdf, geometry="geometry", crs=df.crs)
agreement_gdf

fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
n = len(FAMILY_SCENARIOS)
bins = np.linspace(-n - 0.5, n + 0.5, n * 2 + 2)
cmap = plt.get_cmap("PRGn_r", len(bins) - 1)

ax.add_feature(cfeature.OCEAN, facecolor="0.97")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False

norm = matplotlib.colors.BoundaryNorm(bins, len(bins) - 1)
agreement_gdf.plot(
    "agreement_score",
    transform=ccrs.PlateCarree(),
    ax=ax,
    cmap=cmap,
    norm=norm,
    legend=False,
)
ax.set_extent([20, 320, -52, 58], crs=ccrs.PlateCarree())
    
fig.canvas.draw()
pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.07, pos.y0, 0.02, pos.height])
sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.ax.set_yticks([], minor=False)
cbar.ax.set_yticks([], minor=True)
cbar.set_label('Future trend (and model agreement)', labelpad=10)

fig.savefig(plot_dir / "eapa_pc_trend_model_agreement_map.pdf")